# 🧬 Stable-ChebNet GVAE for Long-Range Protein Structure Analysis
## ENZYMES Dataset — Anomaly Detection via Reconstruction & Representational Error

**Based on:**
- *Return of ChebNet: Understanding and Improving an Overlooked GNN on Long-Range Tasks* (Ali Hariri)
- *Graph learning for capturing long-range dependencies in protein structures* (Ali Hariri, Pierre Vandergheynst)

---
### Architecture Overview
```
Input Graph G(X, A)
      │
      ▼
┌─────────────────────────┐
│  Stable-ChebNet Encoder  │  → Z_node, Z_g
│  X^(l+1) = X^(l) + ε·  │
│  Σ_k T_k(L) X^(l)(W_k-W_k^T-γI)│
└─────────────────────────┘
      │
      ▼
┌─────────────────────────┐
│  Decoder                 │  → X_hat, A_hat
└─────────────────────────┘
      │
      ▼
┌─────────────────────────┐
│  Shared Stable-ChebNet   │  → Z_hat_node, Z_g'
│  Re-Encoder              │
└─────────────────────────┘

Loss = L1 (Reconstruction) + L3 (Representational)
```

## 📦 Cell 1 — Install Dependencies

In [ ]:
# Install all required packages
import subprocess, sys

packages = [
    'torch',
    'torch_geometric',
    'plotly',
    'scipy',
    'scikit-learn',
    'matplotlib',
    'numpy',
    'tqdm',
    'kaleido',
]
for pkg in packages:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

print('✅ All packages installed.')

## 📚 Cell 2 — Imports

In [ ]:
import os
import math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_scatter import scatter

import torch_geometric
from torch_geometric.datasets import TUDataset
from torch_geometric.data import Data, Dataset, InMemoryDataset
from torch_geometric.loader import DataLoader
from torch_geometric.utils import to_dense_adj, get_laplacian, to_scipy_sparse_matrix
from torch_geometric.nn import global_mean_pool, global_add_pool

import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm
import scipy.sparse as sp
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from collections import defaultdict

torch.manual_seed(42)
np.random.seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'🖥️  Device: {device}')
print(f'🔥 PyTorch {torch.__version__} | PyG {torch_geometric.__version__}')

## 🗂️ Cell 3 — Step 1: Load Raw ENZYMES & Store as PyG InMemoryDataset

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# ENZYMES PyG InMemoryDataset
# Mirrors the raw file into dataset/ENZYMES/raw as per PyG convention
# ─────────────────────────────────────────────────────────────────────────────
class ENZYMESDataset(InMemoryDataset):
    """
    Wraps TUDataset ENZYMES with InMemoryDataset for full PyG compatibility.
    Raw files are stored in dataset/ENZYMES/raw/
    Processed tensors stored in dataset/ENZYMES/processed/
    """
    def __init__(self, root='./dataset/ENZYMES', transform=None, pre_transform=None):
        super().__init__(root, transform, pre_transform)
        self.data, self.slices = torch.load(self.processed_paths[0])

    @property
    def raw_file_names(self):
        return ['ENZYMES_A.txt', 'ENZYMES_graph_indicator.txt',
                'ENZYMES_graph_labels.txt', 'ENZYMES_node_attributes.txt',
                'ENZYMES_node_labels.txt']

    @property
    def processed_file_names(self):
        return ['enzymes_data.pt']

    def download(self):
        # Use TUDataset to download into raw folder
        raw_ds = TUDataset(root=self.root, name='ENZYMES', use_node_attr=True)
        print(f'  Downloaded {len(raw_ds)} graphs from TUDataset.')

    def process(self):
        raw_ds = TUDataset(root=self.root, name='ENZYMES', use_node_attr=True)
        data_list = [d for d in raw_ds]
        if self.pre_transform is not None:
            data_list = [self.pre_transform(d) for d in data_list]
        data, slices = self.collate(data_list)
        torch.save((data, slices), self.processed_paths[0])
        print(f'  ✅ Processed {len(data_list)} protein graphs.')


print('📂 Loading ENZYMES dataset...')
os.makedirs('./dataset/ENZYMES', exist_ok=True)
dataset = ENZYMESDataset(root='./dataset/ENZYMES')

print(f'\n📊 ENZYMES Dataset Statistics:')
print(f'   Total graphs   : {len(dataset)}')
print(f'   Node features  : {dataset.num_node_features}')
print(f'   Num classes    : {dataset.num_classes}')

# Quick per-class breakdown
label_counts = defaultdict(int)
num_nodes_list, num_edges_list = [], []
for g in dataset:
    label_counts[g.y.item()] += 1
    num_nodes_list.append(g.num_nodes)
    num_edges_list.append(g.num_edges)

print(f'\n   Graphs per class: {dict(sorted(label_counts.items()))}')
print(f'   Avg nodes/graph : {np.mean(num_nodes_list):.1f}  (min={min(num_nodes_list)}, max={max(num_nodes_list)})')
print(f'   Avg edges/graph : {np.mean(num_edges_list):.1f}')

## 📊 Cell 4 — Step 1b: 3D Visualisation of Protein Graphs (Plotly)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 3D Protein Graph Visualisation
# Node positions generated by PCA on node attributes (18-dim → 3D)
# Node colour = secondary structure type (helix vs sheet via node label)
# ─────────────────────────────────────────────────────────────────────────────

def get_3d_positions(data):
    """Project node features to 3D with PCA for layout."""
    x = data.x.numpy()
    if x.shape[1] >= 3:
        pca = PCA(n_components=3)
        pos = pca.fit_transform(x)
    else:
        pad = np.zeros((x.shape[0], 3 - x.shape[1]))
        pos = np.hstack([x, pad])
    # Scale to unit sphere
    pos = pos / (np.linalg.norm(pos, axis=1, keepdims=True).max() + 1e-8)
    return pos


def visualize_protein_graph_3d(data, title='Protein Graph', show_edges=True):
    pos = get_3d_positions(data)
    edge_index = data.edge_index.numpy()

    # Node colours: use first feature dimension as proxy for helix/sheet type
    node_types = data.x[:, 0].numpy() if data.x.shape[1] > 0 else np.zeros(data.num_nodes)

    # ---- Edge traces ----
    traces = []
    if show_edges:
        ex, ey, ez = [], [], []
        for src, dst in edge_index.T:
            ex += [pos[src, 0], pos[dst, 0], None]
            ey += [pos[src, 1], pos[dst, 1], None]
            ez += [pos[src, 2], pos[dst, 2], None]
        traces.append(go.Scatter3d(
            x=ex, y=ey, z=ez, mode='lines',
            line=dict(color='rgba(120,160,220,0.35)', width=1.5),
            name='Edges (amino-acid / spatial)', hoverinfo='none'
        ))

    # ---- Node trace ----
    traces.append(go.Scatter3d(
        x=pos[:, 0], y=pos[:, 1], z=pos[:, 2],
        mode='markers',
        marker=dict(
            size=6,
            color=node_types,
            colorscale='Viridis',
            colorbar=dict(title='Node type', thickness=12, len=0.6),
            line=dict(color='white', width=0.5),
            opacity=0.9
        ),
        text=[f'Node {i}<br>feat={data.x[i].tolist()[:3]}' for i in range(data.num_nodes)],
        hovertemplate='%{text}<extra></extra>',
        name='Secondary structure elements'
    ))

    fig = go.Figure(data=traces)
    fig.update_layout(
        title=dict(text=title, font=dict(size=16, color='#1a1a2e')),
        scene=dict(
            xaxis=dict(showgrid=False, zeroline=False, showticklabels=False, title='PC1'),
            yaxis=dict(showgrid=False, zeroline=False, showticklabels=False, title='PC2'),
            zaxis=dict(showgrid=False, zeroline=False, showticklabels=False, title='PC3'),
            bgcolor='rgb(240,245,255)'
        ),
        paper_bgcolor='white',
        margin=dict(l=0, r=0, b=0, t=50),
        legend=dict(x=0.01, y=0.99),
        width=750, height=550
    )
    return fig


# ── Show 6 representative proteins (one per class) ──────────────────────────
print('🎨 Rendering 3D protein graph gallery (one graph per enzyme class)...')

class_samples = {}
for g in dataset:
    lbl = g.y.item()
    if lbl not in class_samples:
        class_samples[lbl] = g
    if len(class_samples) == 6:
        break

class_names = {
    0: 'Oxidoreductases', 1: 'Transferases', 2: 'Hydrolases',
    3: 'Lyases', 4: 'Isomerases', 5: 'Ligases'
}

for lbl, g in sorted(class_samples.items()):
    fig = visualize_protein_graph_3d(
        g,
        title=f'Class {lbl}: {class_names[lbl]}  |  Nodes={g.num_nodes}  Edges={g.num_edges}'
    )
    fig.show()
print('✅ 3D visualisation complete.')

## 📊 Cell 5 — Dataset Statistics Dashboard

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Interactive dashboard: node count, edge count, degree distributions
# ─────────────────────────────────────────────────────────────────────────────
degrees_by_class = defaultdict(list)
for g in dataset:
    lbl = g.y.item()
    edge_index = g.edge_index
    deg = torch_geometric.utils.degree(edge_index[0], num_nodes=g.num_nodes).numpy()
    degrees_by_class[lbl].extend(deg.tolist())

fig = make_subplots(
    rows=2, cols=3,
    subplot_titles=[f'{class_names[c]}' for c in range(6)],
    shared_yaxes=False
)
colours = px.colors.qualitative.Plotly
for i, (cls, degs) in enumerate(sorted(degrees_by_class.items())):
    r, c = i // 3 + 1, i % 3 + 1
    fig.add_trace(
        go.Histogram(x=degs, nbinsx=20, name=class_names[cls],
                     marker_color=colours[i], opacity=0.75,
                     showlegend=False),
        row=r, col=c
    )

fig.update_layout(
    title='Node Degree Distributions per Enzyme Class',
    height=550, width=900,
    paper_bgcolor='white', plot_bgcolor='#f8f9fb'
)
fig.show()

# ── Scatter: nodes vs edges coloured by class ────────────────────────────────
nn_list, ne_list, lbl_list = [], [], []
for g in dataset:
    nn_list.append(g.num_nodes)
    ne_list.append(g.num_edges)
    lbl_list.append(g.y.item())

fig2 = px.scatter(
    x=nn_list, y=ne_list,
    color=[class_names[l] for l in lbl_list],
    labels={'x': 'Number of Nodes', 'y': 'Number of Edges', 'color': 'Class'},
    title='ENZYMES — Protein Graph Sizes by Class',
    opacity=0.6, width=800, height=450
)
fig2.update_traces(marker=dict(size=6))
fig2.show()
print('✅ Dataset statistics visualised.')

## 🔬 Cell 6 — Step 3: Dirichlet Energy Analysis — Choosing Best K

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Dirichlet Energy measures feature smoothness:
#   E(X) = tr(X^T L X) / n
# A good K keeps E(X) in a stable, non-collapsing range.
# We simulate K-hop Chebyshev propagation and measure energy at each hop.
# ─────────────────────────────────────────────────────────────────────────────

def get_norm_laplacian(edge_index, num_nodes):
    """Return dense symmetric normalised Laplacian: L = I - D^{-1/2} A D^{-1/2}."""
    edge_weight = torch.ones(edge_index.size(1))
    lap_ei, lap_ew = get_laplacian(edge_index, edge_weight,
                                   normalization='sym', num_nodes=num_nodes)
    L = to_dense_adj(lap_ei, edge_attr=lap_ew, max_num_nodes=num_nodes)[0]
    return L  # [N, N]


def chebyshev_propagate_k_hops(X, L, K):
    """
    Simulate vanilla ChebNet K-hop propagation:
       X_0 = X,  X_1 = L X,  X_k = 2L X_{k-1} - X_{k-2}
    Return list of Dirichlet energies at each hop 0..K.
    """
    n = X.shape[0]
    energies = []

    L_tilde = 2 * L / (L.diagonal().max().clamp(min=1e-8)) - torch.eye(n)

    T_prev = X.clone()           # T_0
    T_curr = L_tilde @ X         # T_1

    def dirichlet(Z, L):
        return (Z.T @ L @ Z).trace().item() / n

    energies.append(dirichlet(T_prev, L))
    if K >= 1:
        energies.append(dirichlet(T_curr, L))

    for _ in range(2, K + 1):
        T_next = 2 * L_tilde @ T_curr - T_prev
        energies.append(dirichlet(T_next, L))
        T_prev = T_curr
        T_curr = T_next

    return energies


def stable_chebnet_propagate_k_hops(X, L, K, eps=0.45, gamma=0.1):
    """
    Simulate Stable-ChebNet K-hop propagation with antisymmetric weights
    (we use identity as proxy weight to isolate the graph operator effect):
       X^(l+1) = X^(l) + eps * T_k(L) X^(l) (I - I^T - gamma*I)
              = X^(l) + eps * T_k(L) X^(l) * (-gamma*I)
    This preserves the ODE stability analysis from Theorem 4.
    """
    n = X.shape[0]
    energies = []

    L_tilde = 2 * L / (L.diagonal().max().clamp(min=1e-8)) - torch.eye(n)

    T_prev = X.clone()
    T_curr = L_tilde @ X

    # Antisymmetric contribution: (W - W^T - γI) with W=I → -γI
    X_running = X.clone()

    def dirichlet(Z, L):
        return (Z.T @ L @ Z).trace().item() / max(n, 1)

    energies.append(dirichlet(X_running, L))

    for k in range(1, K + 1):
        if k == 1:
            Tk = T_curr
        else:
            T_next = 2 * L_tilde @ T_curr - T_prev
            T_prev, T_curr = T_curr, T_next
            Tk = T_curr
        # Stable-ChebNet update
        X_running = X_running + eps * Tk @ X_running * (-gamma)
        energies.append(dirichlet(X_running, L))

    return energies


# ── Compute over a sample of 30 graphs ──────────────────────────────────────
K_MAX = 12
N_SAMPLE = 30
torch.manual_seed(0)

vanilla_energies = np.zeros((N_SAMPLE, K_MAX + 1))
stable_energies  = np.zeros((N_SAMPLE, K_MAX + 1))

indices = torch.randperm(len(dataset))[:N_SAMPLE].tolist()

for idx_sample, data_idx in enumerate(tqdm(indices, desc='Dirichlet energy')):
    g = dataset[data_idx]
    X = g.x.float()   # [N, 18]
    n = g.num_nodes
    if n < 4:
        vanilla_energies[idx_sample] = 0
        stable_energies[idx_sample]  = 0
        continue
    L = get_norm_laplacian(g.edge_index, n)
    # Normalise X
    X = X / (X.norm(dim=1, keepdim=True).clamp(min=1e-8))
    ve = chebyshev_propagate_k_hops(X, L, K_MAX)
    se = stable_chebnet_propagate_k_hops(X, L, K_MAX)
    vanilla_energies[idx_sample] = np.array(ve)
    stable_energies[idx_sample]  = np.array(se)

mean_v = vanilla_energies.mean(axis=0)
std_v  = vanilla_energies.std(axis=0)
mean_s = stable_energies.mean(axis=0)
std_s  = stable_energies.std(axis=0)
k_vals = list(range(K_MAX + 1))

print('✅ Dirichlet energy computation complete.')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Plotly: Dirichlet Energy vs K — Vanilla vs Stable ChebNet
# ─────────────────────────────────────────────────────────────────────────────
fig = go.Figure()

# Vanilla ChebNet band
fig.add_trace(go.Scatter(
    x=k_vals + k_vals[::-1],
    y=list(mean_v + std_v) + list((mean_v - std_v)[::-1]),
    fill='toself', fillcolor='rgba(255,80,80,0.15)',
    line=dict(color='rgba(255,80,80,0)'), name='Vanilla ±1σ'
))
fig.add_trace(go.Scatter(
    x=k_vals, y=mean_v,
    mode='lines+markers', line=dict(color='#e74c3c', width=2.5),
    marker=dict(size=7), name='Vanilla ChebNet'
))

# Stable ChebNet band
fig.add_trace(go.Scatter(
    x=k_vals + k_vals[::-1],
    y=list(mean_s + std_s) + list((mean_s - std_s)[::-1]),
    fill='toself', fillcolor='rgba(40,167,100,0.15)',
    line=dict(color='rgba(40,167,100,0)'), name='Stable ±1σ'
))
fig.add_trace(go.Scatter(
    x=k_vals, y=mean_s,
    mode='lines+markers', line=dict(color='#27ae60', width=2.5, dash='dash'),
    marker=dict(size=7), name='Stable-ChebNet'
))

# Mark optimal K — where vanilla energy stabilises before decay
# We pick K where gradient of vanilla energy flattens (∂E/∂k → 0)
grad = np.abs(np.gradient(mean_v))
best_K = int(np.argmin(grad[2:]) + 2)  # skip k=0,1
fig.add_vline(x=best_K, line=dict(color='#f39c12', width=2, dash='dot'),
              annotation_text=f'Best K = {best_K}',
              annotation_position='top right',
              annotation_font=dict(color='#f39c12', size=13))

fig.update_layout(
    title='Dirichlet Energy vs Chebyshev Order K<br>'
          '<sup>Vanilla ChebNet collapses; Stable-ChebNet preserves long-range signal</sup>',
    xaxis=dict(title='Chebyshev Polynomial Order K', tickmode='linear', tick0=0, dtick=1),
    yaxis=dict(title='Mean Dirichlet Energy E(X)', type='log'),
    paper_bgcolor='white', plot_bgcolor='#f8f9fb',
    legend=dict(x=0.7, y=0.95),
    width=800, height=480
)
fig.show()
print(f'\n🎯 Selected K = {best_K} as optimal Chebyshev order.')
print(f'   (Criterion: smallest |∂E/∂k| beyond k=2 — energy gradient stabilises)')
BEST_K = best_K

## 🏗️ Cell 7 — Step 4: Stable-ChebNet Layer (from Theorem 4)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Stable-ChebNet Layer
#
# From Section 3.3 of the paper (forward Euler + antisymmetric weights):
#   X^(l+1) = X^(l) + ε * Σ_k T_k(L̃) X^(l) (W_k - W_k^T - γI)
#
# Weight antisymmetry: W_eff = W - W^T - γI  →  Jacobian has purely
# imaginary eigenvalues  →  ||J||_2 = 1 + O(ε²)  (Theorem 4)
# ─────────────────────────────────────────────────────────────────────────────

class StableChebNetLayer(nn.Module):
    """
    Single Stable-ChebNet layer implementing:
        X^(l+1) = X^(l) + eps * Σ_{k=0}^K  T_k(L̃) X^(l) (W_k - W_k^T - γ·I)

    Parameters
    ----------
    in_dim   : input feature dimension
    out_dim  : output feature dimension
    K        : Chebyshev polynomial order
    eps      : forward-Euler step size ε
    gamma    : dissipative force γ for numerical stability
    """
    def __init__(self, in_dim: int, out_dim: int, K: int,
                 eps: float = 0.45, gamma: float = 0.1):
        super().__init__()
        self.K     = K
        self.eps   = eps
        self.gamma = gamma

        # One weight matrix per Chebyshev order
        # We store the upper-triangular part only; antisymmetric W_eff = W - W^T
        # For in_dim ≠ out_dim we use a projection first
        self.proj  = nn.Linear(in_dim, out_dim, bias=False) if in_dim != out_dim else nn.Identity()
        self.Ws    = nn.ParameterList([
            nn.Parameter(torch.randn(out_dim, out_dim) * 0.1)
            for _ in range(K + 1)
        ])
        self.bn    = nn.BatchNorm1d(out_dim)

    def _antisymmetric(self, W: torch.Tensor) -> torch.Tensor:
        """W_eff = W - W^T - γ·I  (Theorem 3 construction)"""
        return W - W.t() - self.gamma * torch.eye(W.shape[0], device=W.device)

    @staticmethod
    def _compute_laplacian(edge_index, num_nodes, device):
        """Normalised symmetric graph Laplacian as dense matrix."""
        ei, ew = get_laplacian(edge_index,
                               normalization='sym',
                               num_nodes=num_nodes)
        L = to_dense_adj(ei, edge_attr=ew, max_num_nodes=num_nodes)[0]  # [N,N]
        return L.to(device)

    def forward(self, x: torch.Tensor, edge_index: torch.Tensor) -> torch.Tensor:
        """
        x          : [N, in_dim]
        edge_index : [2, E]
        Returns    : [N, out_dim]
        """
        N = x.shape[0]
        device = x.device

        # Project input to out_dim
        X = self.proj(x)    # [N, out_dim]

        if N == 0 or edge_index.shape[1] == 0:
            return X

        L = self._compute_laplacian(edge_index, N, device)   # [N, N]

        # Normalise L̃ = 2L/λ_max - I  (Chebyshev scaling)
        lambda_max = L.diagonal().max().clamp(min=1e-8)
        L_tilde = 2.0 * L / lambda_max - torch.eye(N, device=device)

        # Chebyshev recurrence: T_0=I, T_1=L̃, T_k=2L̃T_{k-1}-T_{k-2}
        T_prev = torch.eye(N, device=device)
        T_curr = L_tilde

        delta = torch.zeros_like(X)  # accumulate Σ_k T_k(L̃) X W_k_eff

        for k in range(self.K + 1):
            Tk = T_prev if k == 0 else T_curr
            W_eff = self._antisymmetric(self.Ws[k])
            delta = delta + Tk @ X @ W_eff
            # Advance recurrence
            if k >= 1:
                T_next = 2.0 * L_tilde @ T_curr - T_prev
                T_prev = T_curr
                T_curr = T_next

        # Forward-Euler residual update
        X_out = X + self.eps * delta
        X_out = self.bn(X_out)
        return F.elu(X_out)


print('✅ StableChebNetLayer defined.')

## 🏗️ Cell 8 — Step 4b: Stable-ChebNet GVAE (Encoder + Decoder + Re-Encoder)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# StableChebNet-GVAE
#
# Encoder   : 2 × StableChebNet layers  →  µ, logσ  →  Z_node  →  Z_g (pool)
# Decoder   : MLP(Z_node) → X̂  |  inner-product → Â
# Re-Encoder: SHARED encoder weights reused on (X̂, Â_edge_index)
#             → Ẑ_node, Z_g'
#
# Loss = L1 + L3
#   L1 = ||A - Â||_F² + ||X - X̂||_F²              (reconstruction)
#   L_node = (1/|G|) Σ_g (1/|V_g|) Σ_i ||Z_node,i - Ẑ_node,i||²
#   L_graph = (1/|G|) Σ_g ||Z_g - Z_g'||²
#   L3 = L_node + L_graph                            (representational)
# ─────────────────────────────────────────────────────────────────────────────

class StableChebEncoder(nn.Module):
    """Two-layer Stable-ChebNet encoder → µ, logσ per node."""
    def __init__(self, in_dim, hidden_dim, latent_dim, K, eps, gamma):
        super().__init__()
        self.layer1 = StableChebNetLayer(in_dim, hidden_dim, K, eps, gamma)
        self.layer2 = StableChebNetLayer(hidden_dim, hidden_dim, K, eps, gamma)
        self.mu_head    = nn.Linear(hidden_dim, latent_dim)
        self.logvar_head = nn.Linear(hidden_dim, latent_dim)

    def forward(self, x, edge_index, batch):
        h = self.layer1(x, edge_index)      # [N, hidden]
        h = self.layer2(h, edge_index)      # [N, hidden]
        mu     = self.mu_head(h)            # [N, latent]
        logvar = self.logvar_head(h)        # [N, latent]
        # Reparameterisation
        if self.training:
            std = torch.exp(0.5 * logvar).clamp(max=10)
            eps_noise = torch.randn_like(std)
            z_node = mu + eps_noise * std
        else:
            z_node = mu
        # Graph-level: mean pool
        z_g = global_mean_pool(z_node, batch)   # [B, latent]
        return z_node, z_g, mu, logvar


class NodeDecoder(nn.Module):
    """Decode Z_node → X̂ via MLP."""
    def __init__(self, latent_dim, hidden_dim, out_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(latent_dim, hidden_dim),
            nn.ELU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ELU(),
            nn.Linear(hidden_dim, out_dim)
        )

    def forward(self, z_node):
        return self.net(z_node)   # [N, feat_dim]


def decode_adjacency(z_node, batch, threshold=0.5):
    """
    Inner-product decoder:  Â_ij = σ(z_i · z_j)
    Returns edge_index for A_hat (edges above threshold) and the dense Â per graph.
    """
    batch_size = batch.max().item() + 1
    recon_edge_index_list = []
    A_hat_list = []

    offset = 0
    for b in range(batch_size):
        mask = (batch == b)
        z_b  = z_node[mask]                   # [n_b, latent]
        A_b  = torch.sigmoid(z_b @ z_b.t())   # [n_b, n_b]
        A_hat_list.append(A_b)
        # Threshold for discrete edge_index
        adj_b = (A_b > threshold).float()
        adj_b.fill_diagonal_(0)               # no self-loops
        ei = adj_b.nonzero(as_tuple=False).t().contiguous()  # [2, E']
        recon_edge_index_list.append(ei + offset)
        offset += mask.sum().item()

    recon_edge_index = torch.cat(recon_edge_index_list, dim=1)
    return recon_edge_index, A_hat_list


class StableChebGVAE(nn.Module):
    """
    Full Stable-ChebNet GVAE with shared encoder for re-encoding.
    """
    def __init__(self, in_dim, hidden_dim, latent_dim, K,
                 eps=0.45, gamma=0.1):
        super().__init__()
        self.encoder   = StableChebEncoder(in_dim, hidden_dim, latent_dim, K, eps, gamma)
        self.x_decoder = NodeDecoder(latent_dim, hidden_dim, in_dim)
        # Re-encoder shares weights with encoder

    def forward(self, x, edge_index, batch):
        # ── Encode ──────────────────────────────────────────────────────────
        z_node, z_g, mu, logvar = self.encoder(x, edge_index, batch)

        # ── Decode ──────────────────────────────────────────────────────────
        x_hat = self.x_decoder(z_node)              # [N, feat_dim]
        recon_ei, A_hat_list = decode_adjacency(z_node, batch)

        # ── Re-Encode (shared encoder) ───────────────────────────────────────
        # Use reconstructed x_hat and adjacency
        z_hat_node, z_g_prime, _, _ = self.encoder(x_hat.detach(), recon_ei, batch)

        return {
            'z_node':     z_node,
            'z_g':        z_g,
            'z_hat_node': z_hat_node,
            'z_g_prime':  z_g_prime,
            'x_hat':      x_hat,
            'A_hat_list': A_hat_list,
            'mu':         mu,
            'logvar':     logvar,
        }


print('✅ StableChebGVAE defined.')

## 📉 Cell 9 — Step 5: Loss Functions

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Loss Functions (exact formulas from the assignment)
#
# L1 = ||A - Â||_F² + ||X - X̂||_F²
#
# L_node = (1/|G|) Σ_g  (1/|V_g|) Σ_i ||Z_node,i - Ẑ_node,i||²
# L_graph = (1/|G|) Σ_g ||Z_g - Z_g'||²
# L3 = L_node + L_graph
#
# Total: L = L1 + λ·L3
# ─────────────────────────────────────────────────────────────────────────────

def reconstruction_loss(x, x_hat, edge_index, A_hat_list, batch):
    """
    L1 = ||A - Â||_F² + ||X - X̂||_F²
    """
    # Node feature reconstruction
    L_X = F.mse_loss(x_hat, x, reduction='mean')  # ||X - X̂||_F² / N

    # Adjacency reconstruction per graph
    n_graphs = batch.max().item() + 1
    L_A = 0.0
    for b in range(n_graphs):
        mask = (batch == b)
        n_b  = mask.sum().item()
        if n_b == 0:
            continue
        # Ground-truth adjacency
        node_ids = mask.nonzero(as_tuple=False).view(-1)
        # Remap edge_index to local 0..n_b-1 indices
        src, dst = edge_index
        local_mask = mask[src] & mask[dst]
        if local_mask.sum() == 0:
            A_gt = torch.zeros(n_b, n_b, device=x.device)
        else:
            local_src = src[local_mask] - node_ids[0]
            local_dst = dst[local_mask] - node_ids[0]
            A_gt = torch.zeros(n_b, n_b, device=x.device)
            A_gt[local_src.clamp(0, n_b-1), local_dst.clamp(0, n_b-1)] = 1.0

        A_hat_b = A_hat_list[b]
        if A_hat_b.shape[0] != n_b:
            continue
        diff = A_gt - A_hat_b
        L_A += (diff * diff).sum() / (n_b * n_b)

    L_A = L_A / n_graphs
    return L_X + L_A, L_X.item(), L_A


def representational_loss(z_node, z_hat_node, z_g, z_g_prime, batch):
    """
    L_node = (1/|G|) Σ_g  (1/n_g) Σ_i ||Z_node,i - Ẑ_node,i||²
    L_graph = (1/|G|) Σ_g ||Z_g - Z_g'||²
    L3 = L_node + L_graph
    """
    n_graphs = batch.max().item() + 1

    # Node-level
    L_node = 0.0
    for b in range(n_graphs):
        mask = (batch == b)
        n_b  = mask.sum().item()
        if n_b == 0:
            continue
        diff = z_node[mask] - z_hat_node[mask]
        L_node += (diff * diff).sum() / n_b
    L_node = L_node / n_graphs

    # Graph-level
    diff_g = z_g - z_g_prime                        # [B, latent]
    L_graph = (diff_g * diff_g).sum(dim=-1).mean()  # scalar

    return L_node + L_graph, L_node, L_graph.item()


print('✅ Loss functions defined.')

## 🚀 Cell 10 — Step 4c: Instantiate Model & Train

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Hyperparameters  (following config_StableCheb.json from the repo)
# ─────────────────────────────────────────────────────────────────────────────
IN_DIM      = dataset.num_node_features   # 18 for ENZYMES
HIDDEN_DIM  = 64
LATENT_DIM  = 32
K           = BEST_K                      # from Dirichlet energy analysis
EPS         = 0.45                        # step_size from config
GAMMA       = 0.10                        # dissipative_force from config
LR          = 1e-3
EPOCHS      = 80                          # reduced for notebook speed
BATCH_SIZE  = 32
LAMBDA_REP  = 0.5                         # weight for representational loss

print(f'📐 Model config:')
print(f'   in_dim={IN_DIM}  hidden={HIDDEN_DIM}  latent={LATENT_DIM}')
print(f'   K={K}  ε={EPS}  γ={GAMMA}  lr={LR}  epochs={EPOCHS}')

# ── Train/Val/Test split ─────────────────────────────────────────────────────
torch.manual_seed(42)
perm = torch.randperm(len(dataset))
n_train = int(0.7 * len(dataset))
n_val   = int(0.15 * len(dataset))

train_data = [dataset[i.item()] for i in perm[:n_train]]
val_data   = [dataset[i.item()] for i in perm[n_train:n_train+n_val]]
test_data  = [dataset[i.item()] for i in perm[n_train+n_val:]]

train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True,  drop_last=False)
val_loader   = DataLoader(val_data,   batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_data,  batch_size=BATCH_SIZE, shuffle=False)

print(f'\n   Train: {len(train_data)}  Val: {len(val_data)}  Test: {len(test_data)}')

# ── Model & Optimizer ────────────────────────────────────────────────────────
model = StableChebGVAE(
    in_dim=IN_DIM, hidden_dim=HIDDEN_DIM, latent_dim=LATENT_DIM,
    K=K, eps=EPS, gamma=GAMMA
).to(device)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'\n   Trainable parameters: {total_params:,}')

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=10, min_lr=1e-5, verbose=False
)
print('✅ Model ready.')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Training Loop
# ─────────────────────────────────────────────────────────────────────────────
history = {'train_total':[], 'train_L1':[], 'train_L3':[],
           'val_total':[],   'val_L1':[],   'val_L3':[]}

best_val_loss = float('inf')
best_state    = None

for epoch in range(1, EPOCHS + 1):
    # ── Train ────────────────────────────────────────────────────────────────
    model.train()
    total_loss = total_L1 = total_L3 = 0.0
    n_batches  = 0

    for batch_data in train_loader:
        batch_data = batch_data.to(device)
        x, ei, bat = batch_data.x.float(), batch_data.edge_index, batch_data.batch

        optimizer.zero_grad()
        out = model(x, ei, bat)

        L1, lx, la = reconstruction_loss(x, out['x_hat'], ei, out['A_hat_list'], bat)
        L3, lnd, lgr = representational_loss(
            out['z_node'], out['z_hat_node'], out['z_g'], out['z_g_prime'], bat
        )
        loss = L1 + LAMBDA_REP * L3
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        total_loss += loss.item()
        total_L1   += L1.item()
        total_L3   += L3.item()
        n_batches  += 1

    history['train_total'].append(total_loss / n_batches)
    history['train_L1'].append(total_L1   / n_batches)
    history['train_L3'].append(total_L3   / n_batches)

    # ── Validate ─────────────────────────────────────────────────────────────
    model.eval()
    v_total = v_L1 = v_L3 = 0.0
    v_n = 0
    with torch.no_grad():
        for vd in val_loader:
            vd  = vd.to(device)
            x_v = vd.x.float()
            out_v = model(x_v, vd.edge_index, vd.batch)
            vL1, _, _ = reconstruction_loss(x_v, out_v['x_hat'], vd.edge_index,
                                            out_v['A_hat_list'], vd.batch)
            vL3, _, _ = representational_loss(
                out_v['z_node'], out_v['z_hat_node'],
                out_v['z_g'], out_v['z_g_prime'], vd.batch
            )
            vloss = vL1 + LAMBDA_REP * vL3
            v_total += vloss.item()
            v_L1    += vL1.item()
            v_L3    += vL3.item()
            v_n     += 1

    v_avg = v_total / max(v_n, 1)
    history['val_total'].append(v_avg)
    history['val_L1'].append(v_L1 / max(v_n, 1))
    history['val_L3'].append(v_L3 / max(v_n, 1))

    scheduler.step(v_avg)

    if v_avg < best_val_loss:
        best_val_loss = v_avg
        best_state    = {k: v.cpu().clone() for k, v in model.state_dict().items()}

    if epoch % 10 == 0 or epoch == 1:
        print(f'Epoch {epoch:3d}/{EPOCHS}  '
              f'Train={history["train_total"][-1]:.4f} '
              f'(L1={history["train_L1"][-1]:.4f}, L3={history["train_L3"][-1]:.4f})  '
              f'Val={v_avg:.4f}')

# Restore best weights
model.load_state_dict(best_state)
print(f'\n✅ Training complete. Best val loss: {best_val_loss:.4f}')

## 📈 Cell 11 — Training Curves

In [ ]:
epochs_x = list(range(1, EPOCHS + 1))

fig = make_subplots(rows=1, cols=3,
                    subplot_titles=['Total Loss (L1 + λL3)', 'L1: Reconstruction', 'L3: Representational'])
keys = [('train_total','val_total'), ('train_L1','val_L1'), ('train_L3','val_L3')]

for col, (tr_key, va_key) in enumerate(keys, 1):
    fig.add_trace(go.Scatter(x=epochs_x, y=history[tr_key], mode='lines',
                             name='Train', line=dict(color='#3498db', width=2),
                             showlegend=(col == 1)), row=1, col=col)
    fig.add_trace(go.Scatter(x=epochs_x, y=history[va_key], mode='lines',
                             name='Val', line=dict(color='#e74c3c', width=2, dash='dash'),
                             showlegend=(col == 1)), row=1, col=col)

fig.update_layout(
    title='Stable-ChebNet GVAE — Training Curves on ENZYMES',
    height=380, width=1000,
    paper_bgcolor='white', plot_bgcolor='#f8f9fb',
    legend=dict(x=0.01, y=0.99)
)
fig.show()
print('✅ Training curves displayed.')

## 🔮 Cell 12 — Step 6: Z_g vs Z_g' Latent Space Visualisation

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Collect Z_g (encoder) and Z_g' (re-encoder) for the full test set
# Visualise with PCA → 2D scatter, coloured by class
# ─────────────────────────────────────────────────────────────────────────────
model.eval()
Zg_list, Zg_prime_list, labels_list = [], [], []

with torch.no_grad():
    for td in test_loader:
        td = td.to(device)
        out = model(td.x.float(), td.edge_index, td.batch)
        Zg_list.append(out['z_g'].cpu())
        Zg_prime_list.append(out['z_g_prime'].cpu())
        labels_list.append(td.y.cpu())

Zg_all       = torch.cat(Zg_list,       dim=0).numpy()   # [N_test, latent]
Zg_prime_all = torch.cat(Zg_prime_list, dim=0).numpy()
labels_all   = torch.cat(labels_list,   dim=0).numpy()

# PCA to 2D for both jointly
combined = np.vstack([Zg_all, Zg_prime_all])
pca2 = PCA(n_components=2, random_state=42)
combined_2d = pca2.fit_transform(combined)
n_test = len(Zg_all)
Zg_2d       = combined_2d[:n_test]
Zg_prime_2d = combined_2d[n_test:]

def make_zg_scatter(pts, labels, title, marker_symbol='circle'):
    fig = go.Figure()
    for cls in sorted(set(labels)):
        mask = labels == cls
        fig.add_trace(go.Scatter(
            x=pts[mask, 0], y=pts[mask, 1],
            mode='markers',
            name=class_names[cls],
            marker=dict(size=9, symbol=marker_symbol, opacity=0.75,
                        line=dict(width=0.5, color='white'))
        ))
    fig.update_layout(
        title=title, xaxis_title='PC1', yaxis_title='PC2',
        width=620, height=480,
        paper_bgcolor='white', plot_bgcolor='#f8f9fb',
        legend=dict(title='Enzyme Class')
    )
    return fig

fig_zg = make_zg_scatter(Zg_2d, labels_all,
    title='Z_g from Encoder — PCA 2D (test set)', marker_symbol='circle')
fig_zg.show()

fig_zgp = make_zg_scatter(Zg_prime_2d, labels_all,
    title="Z_g' from Re-Encoder — PCA 2D (test set)", marker_symbol='diamond')
fig_zgp.show()

# ── Overlay comparison ───────────────────────────────────────────────────────
fig_ov = go.Figure()
colours = px.colors.qualitative.Plotly
for cls in sorted(set(labels_all)):
    mask = labels_all == cls
    c = colours[cls % len(colours)]
    fig_ov.add_trace(go.Scatter(
        x=Zg_2d[mask, 0], y=Zg_2d[mask, 1],
        mode='markers', name=f'Z_g  {class_names[cls]}',
        marker=dict(size=8, color=c, symbol='circle', opacity=0.6)
    ))
    fig_ov.add_trace(go.Scatter(
        x=Zg_prime_2d[mask, 0], y=Zg_prime_2d[mask, 1],
        mode='markers', name=f"Z_g' {class_names[cls]}",
        marker=dict(size=8, color=c, symbol='x', opacity=0.6)
    ))

fig_ov.update_layout(
    title="Z_g (○) vs Z_g' (✕) — Encoder vs Re-Encoder Comparison",
    xaxis_title='PC1', yaxis_title='PC2',
    width=800, height=520,
    paper_bgcolor='white', plot_bgcolor='#f8f9fb'
)
fig_ov.show()
print('✅ Latent space visualisation complete.')

## 🧬 Cell 13 — Step 8: Original vs Reconstructed Protein Graphs (3D Plotly)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Side-by-side 3D: Original G vs Reconstructed Ĝ
# Node positions: PCA on original features for both (shared layout)
# Node colour: feature magnitude ∥x_i∥
# ─────────────────────────────────────────────────────────────────────────────

def compare_graphs_3d(orig_data, x_hat_np, recon_ei, title_prefix=''):
    x_orig = orig_data.x.numpy()
    ei_orig = orig_data.edge_index.numpy()

    # Shared 3D layout from original features
    if x_orig.shape[0] >= 3 and x_orig.shape[1] >= 3:
        pca = PCA(n_components=3)
        pos = pca.fit_transform(x_orig)
    elif x_orig.shape[1] == 2:
        pos = np.hstack([x_orig, np.zeros((x_orig.shape[0], 1))])
    else:
        pos = np.hstack([x_orig[:, :1],
                         np.zeros((x_orig.shape[0], 1)),
                         np.zeros((x_orig.shape[0], 1))])
    pos = pos / (np.abs(pos).max() + 1e-8)
    n = x_orig.shape[0]

    def make_3d_traces(edge_np, node_colors, node_label, edge_color, edge_name):
        ex, ey, ez = [], [], []
        for s, d in edge_np.T:
            if s < n and d < n:
                ex += [pos[s, 0], pos[d, 0], None]
                ey += [pos[s, 1], pos[d, 1], None]
                ez += [pos[s, 2], pos[d, 2], None]

        edge_trace = go.Scatter3d(
            x=ex, y=ey, z=ez, mode='lines',
            line=dict(color=edge_color, width=2), name=edge_name, hoverinfo='none'
        )
        node_trace = go.Scatter3d(
            x=pos[:, 0], y=pos[:, 1], z=pos[:, 2],
            mode='markers',
            marker=dict(size=7, color=node_colors, colorscale='Plasma',
                        line=dict(color='white', width=0.5), opacity=0.9),
            name=node_label
        )
        return [edge_trace, node_trace]

    orig_colors  = np.linalg.norm(x_orig,   axis=1)
    recon_colors = np.linalg.norm(x_hat_np, axis=1) if x_hat_np is not None else orig_colors

    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=[f'{title_prefix} — Original G', f'{title_prefix} — Reconstructed Ĝ'],
        specs=[[{'type': 'scatter3d'}, {'type': 'scatter3d'}]]
    )

    for tr in make_3d_traces(ei_orig, orig_colors,
                              'Original nodes', 'rgba(70,130,220,0.35)', 'Original edges'):
        fig.add_trace(tr, row=1, col=1)

    recon_ei_np = recon_ei.cpu().numpy() if torch.is_tensor(recon_ei) else recon_ei
    for tr in make_3d_traces(recon_ei_np, recon_colors,
                              'Reconstructed nodes', 'rgba(230,100,50,0.35)', 'Reconstructed edges'):
        fig.add_trace(tr, row=1, col=2)

    scene_cfg = dict(
        xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
        yaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
        zaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
        bgcolor='rgb(242,248,255)'
    )
    fig.update_layout(
        scene=scene_cfg, scene2=scene_cfg,
        height=520, width=1000,
        paper_bgcolor='white',
        margin=dict(l=0, r=0, t=50, b=0)
    )
    return fig


# ── Visualise 4 test graphs ──────────────────────────────────────────────────
print('🎨 Rendering Original vs Reconstructed graphs (4 examples)...')
model.eval()

SHOW_N = 4
shown  = 0
with torch.no_grad():
    for td in test_loader:
        td   = td.to(device)
        xf   = td.x.float()
        out  = model(xf, td.edge_index, td.batch)
        x_hat_all   = out['x_hat'].cpu().numpy()
        recon_ei_all = out['x_hat'].cpu()  # placeholder, use A_hat derived ei
        # reconstruct per graph
        recon_full_ei, _ = decode_adjacency(out['z_node'], td.batch)

        batch_np = td.batch.cpu().numpy()
        for b in range(td.batch.max().item() + 1):
            if shown >= SHOW_N:
                break
            mask = batch_np == b
            n_b  = mask.sum()
            if n_b < 4:
                continue

            # Gather single-graph data
            offset  = int(batch_np[:np.where(mask)[0][0]].sum() if np.where(mask)[0][0] > 0 else 0)
            node_ids = np.where(mask)[0]
            start_id = node_ids[0]

            x_orig_b = td.x[mask].cpu()
            x_hat_b  = x_hat_all[mask]

            # Get original edge_index for this graph
            src, dst = td.edge_index.cpu().numpy()
            local_e  = (td.batch.cpu().numpy()[src] == b)
            ei_b     = td.edge_index[:, local_e].cpu() - start_id

            # Get reconstructed edge_index
            src2, dst2 = recon_full_ei.cpu().numpy()
            valid2 = ((src2 >= start_id) & (src2 < start_id + n_b) &
                      (dst2 >= start_id) & (dst2 < start_id + n_b))
            recon_ei_b = torch.tensor(np.stack([src2[valid2]-start_id,
                                                dst2[valid2]-start_id]))

            g_tmp = Data(x=x_orig_b.float(), edge_index=ei_b)
            lbl   = td.y[b].item() if td.y.dim() == 1 else td.y[mask][0].item()

            fig = compare_graphs_3d(
                g_tmp, x_hat_b, recon_ei_b,
                title_prefix=f'Graph {shown+1} | Class: {class_names.get(lbl, lbl)} | N={n_b}'
            )
            fig.show()
            shown += 1

        if shown >= SHOW_N:
            break

print('✅ Reconstruction visualisations complete.')

## 📊 Cell 14 — Reconstruction Quality Metrics

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Per-class reconstruction error and anomaly score = total loss
# ─────────────────────────────────────────────────────────────────────────────
model.eval()
class_errors = defaultdict(list)
graph_scores = []  # (loss, label)

with torch.no_grad():
    for td in test_loader:
        td = td.to(device)
        xf = td.x.float()
        out = model(xf, td.edge_index, td.batch)

        L1, _, _ = reconstruction_loss(xf, out['x_hat'], td.edge_index,
                                       out['A_hat_list'], td.batch)
        L3, _, _ = representational_loss(
            out['z_node'], out['z_hat_node'],
            out['z_g'], out['z_g_prime'], td.batch
        )

        # Per-graph scores: ||Z_g - Z_g'||² as anomaly proxy
        per_graph = ((out['z_g'] - out['z_g_prime'])**2).sum(dim=1).cpu().numpy()
        ys = td.y.cpu().numpy()
        for i, (sc, lbl) in enumerate(zip(per_graph, ys)):
            class_errors[int(lbl)].append(float(sc))
            graph_scores.append((float(sc), int(lbl)))

# ── Per-class box plot ───────────────────────────────────────────────────────
fig = go.Figure()
colours = px.colors.qualitative.Plotly
for cls in sorted(class_errors.keys()):
    fig.add_trace(go.Box(
        y=class_errors[cls],
        name=class_names[cls],
        marker_color=colours[cls % len(colours)],
        boxmean='sd',
        notched=True
    ))

fig.update_layout(
    title='Anomaly Score (||Z_g − Z_g\'||²) per Enzyme Class<br>'
          '<sup>Higher = more anomalous representation gap</sup>',
    yaxis_title='Representational Gap ||Z_g − Z_g\'||²',
    xaxis_title='Enzyme Class',
    paper_bgcolor='white', plot_bgcolor='#f8f9fb',
    height=480, width=850
)
fig.show()

# Print summary
print('\n📊 Mean anomaly score per class:')
for cls in sorted(class_errors.keys()):
    vals = class_errors[cls]
    print(f'   {class_names[cls]:20s}: {np.mean(vals):.4f} ± {np.std(vals):.4f}')
print('\n✅ Anomaly analysis complete.')

## 📐 Cell 15 — Summary & Theoretical Grounding

In [ ]:
print('=' * 70)
print('  STABLE-CHEBNET GVAE — ENZYMES PROTEIN STRUCTURE ANALYSIS')
print('=' * 70)

print(f'''
📐 ARCHITECTURE SUMMARY
───────────────────────
Layer update (Theorem 4 — Stable-ChebNet):
  X^(l+1) = X^(l) + ε · Σ_k T_k(L̃) X^(l) (W_k − W_k^T − γI)

  • Antisymmetry W_k − W_k^T  →  Jacobian eigenvalues purely imaginary (Theorem 3)
  • Forward-Euler step ε       →  ||J||₂ = 1 + O(ε²)  (Theorem 4: no exp. growth)
  • K = {BEST_K}  (selected via Dirichlet energy analysis)
  • ε = {EPS},  γ = {GAMMA}

🔢 MODEL PARAMETERS
────────────────────
  In dim    : {IN_DIM}   (ENZYMES node attributes)
  Hidden    : {HIDDEN_DIM}
  Latent    : {LATENT_DIM}
  Total     : {total_params:,} trainable parameters

📉 LOSS FUNCTIONS
──────────────────
  L1 = ||A − Â||_F² + ||X − X̂||_F²         (reconstruction)
  L_node = (1/|G|) Σ_g (1/n_g) Σ_i ||Z_i − Ẑ_i||²
  L_graph = (1/|G|) Σ_g ||Z_g − Z_g'||²
  L3 = L_node + L_graph                       (representational)
  L_total = L1 + λ·L3,  λ = {LAMBDA_REP}

📊 RESULTS
───────────
  Best Val Loss : {best_val_loss:.4f}
  Test graphs   : {len(test_data)}

  Anomaly score = ||Z_g − Z_g'||²:
  A graph where encoder and re-encoder disagree has high anomaly score,
  indicating the reconstructed structure does not preserve the original
  protein's long-range spectral properties.
''')
print('=' * 70)